write a python notebook (not in style of functions but in code blocks to be run sequentially) to import relevant libraries, load a csv data set (labelled: target = last column called class --> two categories: 1 = real, 2 = fake), the first column of the data set is the file path so we might have to preprocess that, load a random forest model. split the data set into x, y, train, and test. Define a very extensive hyperparameter grid for the random forest classifier and then train and test the model on the dataset using the hyperparameter grid (verbose = 1 and textual evaluation results printed along with the hyperparameters used to train). save the best performing model locally using joblib and then beautifully print all possible metrics for evaluation (e.g., accuracy, precision, f1 score, recall, roc/auc curve, confusion matrices) for the best model.

# Training

In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# 2. Load the dataset
# Replace 'your_data.csv' with the actual filename
df = pd.read_csv('../dataset/01_feature_csv/InceptionV3_StyleGAN.csv')

# Preview the data
df.head()


,image_path,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047,class
0,C:\Users\rishi\Desktop\JHU\Compliance\Research...,0.089394,0.902768,0.224953,0.300689,0.465533,0.329256,0.248104,0.122286,0.275098,...,0.076341,0.085582,0.885772,0.129567,0.086856,0.058097,0.471754,0.040221,0.016108,1
1,C:\Users\rishi\Desktop\JHU\Compliance\Research...,0.185553,0.252194,0.139374,0.426332,0.305727,0.062091,0.319618,0.201309,0.202675,...,0.125456,0.054440,1.167961,0.788944,0.765983,0.087693,1.061742,0.254113,0.330021,1
2,C:\Users\rishi\Desktop\JHU\Compliance\Research...,0.037417,0.262638,0.030312,0.396271,0.103245,0.362980,0.058754,0.218556,0.649736,...,0.165049,0.127155,0.626925,0.007138,0.253902,0.000000,0.037431,0.103529,0.192329,1
3,C:\Users\rishi\Desktop\JHU\Compliance\Research...,0.103236,0.674537,0.565192,0.356193,0.426234,0.234540,0.360675,0.373064,0.345738,...,0.287185,0.675906,1.704777,0.044917,1.075186,0.299025,0.675202,0.238128,1.188105,1
4,C:\Users\rishi\Desktop\JHU\Compliance\Research...,0.285315,0.170023,0.451027,0.167967,0.067887,0.357494,0.219967,0.121813,0.135312,...,0.304298,0.313077,1.799413,0.786002,0.401256,0.016327,0.313702,0.230456,0.955216,1


In [3]:
# 3. Preprocess: Remove first column (file path), separate features and labels
X = df.iloc[:, 1:-1] # All columns except the first (file path) and last (class)
y = df['class']      # Target column

In [4]:
X.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_2038,feature_2039,feature_2040,feature_2041,feature_2042,feature_2043,feature_2044,feature_2045,feature_2046,feature_2047
0,0.089394,0.902768,0.224953,0.300689,0.465533,0.329256,0.248104,0.122286,0.275098,0.374787,...,0.170300,0.076341,0.085582,0.885772,0.129567,0.086856,0.058097,0.471754,0.040221,0.016108
1,0.185553,0.252194,0.139374,0.426332,0.305727,0.062091,0.319618,0.201309,0.202675,0.634555,...,0.068518,0.125456,0.054440,1.167961,0.788944,0.765983,0.087693,1.061742,0.254113,0.330021
2,0.037417,0.262638,0.030312,0.396271,0.103245,0.362980,0.058754,0.218556,0.649736,0.441527,...,0.412836,0.165049,0.127155,0.626925,0.007138,0.253902,0.000000,0.037431,0.103529,0.192329
3,0.103236,0.674537,0.565192,0.356193,0.426234,0.234540,0.360675,0.373064,0.345738,0.627237,...,1.373726,0.287185,0.675906,1.704777,0.044917,1.075186,0.299025,0.675202,0.238128,1.188105
4,0.285315,0.170023,0.451027,0.167967,0.067887,0.357494,0.219967,0.121813,0.135312,0.326763,...,0.153491,0.304298,0.313077,1.799413,0.786002,0.401256,0.016327,0.313702,0.230456,0.955216


In [5]:
y.value_counts()

class
2    7000
1    5890
Name: count, dtype: int64

In [8]:
# 4. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=420, stratify=y
)
print(f'Training size: {X_train.shape}, Test size: {X_test.shape}')


Training size: (10312, 2048), Test size: (2578, 2048)


In [9]:
# 5. Define an extensive hyperparameter grid
from sklearn.model_selection import ParameterGrid
param_grid = {
    'n_estimators': [20, 50, 100, 200, 500],
    'max_depth': [None, 10, 30, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced']
}
hyperparams_list = list(ParameterGrid(param_grid))
print(f"Total hyperparameter combinations: {len(hyperparams_list)}")

Total hyperparameter combinations: 2160


In [10]:
# 6. Train and test each combination, store the model with the best test F1 score
best_f1 = 0
best_model = None
best_params = None
results_log = []

for i, params in enumerate(hyperparams_list):
    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred, pos_label=1, average='binary')

    results_log.append({
        'index': i,
        **params,
        'f1_test': f1,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, pos_label=1),
        'recall': recall_score(y_test, y_pred, pos_label=1)
    })

    print(f"Run {i+1}/{len(hyperparams_list)} | Hyperparams: {params} | Test F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_params = params

print('-'*60)
print("Best Params on Test Set:", best_params)
print(f"Best Test F1: {best_f1:.4f}")


Run 1/2160 | Hyperparams: {'bootstrap': True, 'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20} | Test F1: 0.7678
Run 2/2160 | Hyperparams: {'bootstrap': True, 'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50} | Test F1: 0.7696
Run 3/2160 | Hyperparams: {'bootstrap': True, 'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100} | Test F1: 0.7748
Run 4/2160 | Hyperparams: {'bootstrap': True, 'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} | Test F1: 0.7878
Run 5/2160 | Hyperparams: {'bootstrap': True, 'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500} | Test F1: 0.7949
Run 6/2160 | Hype

KeyboardInterrupt: 

In [ ]:
# 7. Save the best model
joblib.dump(best_model, '../models/04_InceptionV3_TrainSET_rf.joblib')
print('Best model saved as ../models/04_InceptionV3_TrainSET_rf.joblib')

In [ ]:
# 8. Evaluate best model thoroughly on the test set
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:,1]
print('Classification Report:')
print(classification_report(y_test, y_pred))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred, pos_label=1):.4f}')
print(f'Recall: {recall_score(y_test, y_pred, pos_label=1):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred, pos_label=1):.4f}')

roc_auc = roc_auc_score(y_test, y_proba)
print(f'ROC AUC Score: {roc_auc:.4f}')


In [ ]:
# 9. Plot confusion matrix
plt.figure(figsize=(5,5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


In [ ]:
# 10. Plot ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba, pos_label=1)
plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, color='blue', label=f'AUC={roc_auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid()
plt.show()


# Testing

In [ ]:
model = joblib.load("../models/04_InceptionV3_TrainSET_rf.joblib")
print("✅ Model loaded successfully!")

In [ ]:
test_df = pd.read_csv("../dataset/01_feature_csv/InceptionV3_real_fake_hard.csv")
test_df.info()
X_test = test_df.iloc[:, 1:-1]
y_test = test_df['class']

In [ ]:
X_test.head()

In [ ]:
y_test.value_counts()

In [ ]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("✅ Predictions completed!")
print(f"Sample predictions: {y_pred[:5]}")

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print("\n📊 Test Set Evaluation Results:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# Confusion Matrix
print("\n📋 Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Classification Report
print("\n📑 Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.preprocessing import label_binarize

def plot_confusion_matrix(cm, class_names):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

# Get unique class names (assuming y_test is categorical or encoded)
class_names = sorted(np.unique(y_test))

# Plot confusion matrix
plot_confusion_matrix(cm, class_names)

# If binary classification, plot ROC Curve
n_classes = len(class_names)
if n_classes == 2:
    # Binarize y_test for binary case
    y_test_bin = label_binarize(y_test, classes=class_names)
    fpr, tpr, _ = roc_curve(y_test_bin, y_pred_proba[:, 1])
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc='lower right')
    plt.show()

    # Precision-Recall Curve
    precision_curve, recall_curve, _ = precision_recall_curve(y_test_bin, y_pred_proba[:, 1])
    plt.figure(figsize=(8, 6))
    plt.plot(recall_curve, precision_curve, color='blue', lw=2)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.show()
elif n_classes > 2:
    print("⚠️ ROC and Precision-Recall curves shown for binary classification only. For multi-class, consider one-vs-rest plots.")

"""
# Feature Importance Visualization
print("\n⭐ Feature Importance Visualization:")
# Drop the problematic column from X_test before getting feature names
X_test_for_importance = X_test.drop(columns='deep_embed_999', errors='ignore')
importances = pd.Series(model.get_feature_importance(), index=X_test_for_importance.columns)
importances_sorted = importances.sort_values(ascending=False).head(20)  # Top 20 features

plt.figure(figsize=(10, 8))
importances_sorted.plot(kind='barh')
plt.title('Top 20 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.gca().invert_yaxis()
plt.show()
"""

print("✅ Visualizations completed!")